In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset
from fundus_vessels_toolkit.segment_to_graph.models.losses import VBranchDigraphMiner
from fundus_vessels_toolkit.segment_to_graph.models.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.models.trainer import DigraphGNNTrainer
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.tree import tree_connected_components

vscode_theme()


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
import datetime

PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]
opts = dict(resize_to=1024, root="tmp/DATA", ignore_recent=datetime.datetime(2026, 2, 19))
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, **opts)
train_set, val_set, test_set = dataset.split_loaders(train_ratio=0.7, val_ratio=0.15)

Found 215 branch digraphs...


Processing...
Done!
Preloading dataset: 100%|██████████| 215/215 [00:22<00:00,  9.43it/s]


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [5]:
model = DigraphGNNTrainer.load_from_checkpoint("tmp/invalid.ckpt").model.cuda().eval()

In [77]:
ID = 23
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(test_set.get(ID).cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

gt_digraph, _, od_yx, _ = test_set.get_sample(ID)
assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


0.944954128440367
0.8287461773700305


In [78]:
from pytorch_metric_learning import losses, distances


pairs = VBranchDigraphMiner(triplet=False)(out)
triplet = VBranchDigraphMiner(triplet=True)(out)

CosSim = distances.CosineSimilarity()
(
    losses.ContrastiveLoss(distance=CosSim, pos_margin=1, neg_margin=0)(out.b1_embedding, indices_tuple=pairs),
    losses.TripletMarginLoss()(out.b1_embedding, indices_tuple=triplet),
)

(tensor(0.9451, device='cuda:0'), tensor(0.2599, device='cuda:0'))

In [8]:
m, pred_tree = test_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
)
m

[ WARN:0@27.015] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">GT Tree: 052_N</h3>'), HTML(value='<h3 style="te…

In [22]:
m.views[0].goto(pred_tree.branch(177).midpoint().xy, scale=3)

In [10]:
pred_tree.branch_tree[15]

np.int64(345)

In [11]:
B_ID = 144
pred_av[B_ID], out.av_logit[B_ID], pred_parent[B_ID], out.max_parent()[B_ID]

(tensor(-0.7555, device='cuda:0'),
 tensor(7.6354, device='cuda:0'),
 tensor(-1, device='cuda:0'),
 tensor(130, device='cuda:0'))

In [12]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=216, sort_by_p=True).round(3)

,b0,b1,tip0,tip1,line_p,av_p,total_p,b0_dir_p,b1_dir_p
0,130,216,1,0,0.894,0.994,2.883,0.999,0.991
1,144,216,1,1,0.833,0.993,2.329,0.996,0.009
2,126,216,1,0,0.007,0.994,1.996,1.000,0.991
3,96,216,0,0,0.222,0.776,1.958,0.928,0.991
4,144,216,0,0,0.097,0.993,1.588,0.004,0.991
5,98,216,0,0,0.010,0.990,1.573,0.155,0.991
6,124,216,0,0,0.050,0.950,1.559,0.127,0.991
7,121,216,0,0,0.067,0.933,1.522,0.053,0.991
8,210,216,1,0,0.007,0.993,1.498,0.004,0.991
9,337,216,1,0,0.035,0.965,1.496,0.001,0.991


In [13]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []
node_ratio = []
branch_ratio = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        gt_digraph, _, od_yx, _ = eval_set.get_sample(i)
        assert gt_digraph.branch_dir_p is not None
        assert gt_digraph.graph is not None, "Graph must be loaded to infer tree"

        node_ratio += [gt_digraph.graph.node_count / eval_set.graphs[i].node_count]
        branch_ratio += [gt_digraph.graph.branch_count / eval_set.graphs[i].branch_count]

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(od_yx)

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(eval_set.get(i).cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

 58%|█████▊    | 19/33 [00:05<00:05,  2.65it/s]/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/vbranch_digraph.py:814: UserWarning: Impossible to solve the simplified optimal tree: No maximum spanning arborescence in G.. 
 Fallback to maximum branching.
  branch_parents, branch_dir = solve_line_digraph_approx(
100%|██████████| 33/33 [00:13<00:00,  2.52it/s]


In [14]:
np.mean(node_ratio), np.mean(branch_ratio)

(np.float64(1.485672855640502), np.float64(1.4890490070562852))

In [15]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.8979209012782241),
 np.float64(0.8948540216857692),
 np.float64(0.8581348769400229))

In [16]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9810545959760755),
 np.float64(0.9772750721608982),
 np.float64(0.9452025534803274))

In [17]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9588152992203596),
 np.float64(0.9506956017766368),
 np.float64(0.953922236768025))

In [18]:
np.array(opti_av_acc)[23], np.array(pred_av_acc)[23]

(np.float64(0.8287461773700305), np.float64(0.944954128440367))

In [19]:
(np.array(pred_parent_acc) - np.array(pred_parent_acc)).argsort()[::-1]

array([32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1,  0])

In [20]:
name

['g_039',
 'g_016',
 'g_011',
 'g_050',
 'g_031',
 'g_007',
 'g_043',
 'g_003',
 '20060407_45107_0200_PP',
 '20051021_39222_0100_PP',
 '20051205_35354_0400_PP',
 '20051205_57988_0400_PP',
 '20051205_35305_0400_PP',
 '20051201_38211_0400_PP',
 '20060410_41767_0200_PP',
 '20051020_64007_0100_PP',
 '20051208_39438_0400_PP',
 '20051212_36548_0400_PP',
 '098_D',
 '015_N',
 '002_N',
 '084_D',
 '040_G',
 '052_N',
 '069_N',
 '093_N',
 '032_A',
 '078_D',
 '013_A',
 '081_D',
 '076_D',
 '086_N',
 '028_G']